# Lab: Einstein Summation in TensorFlow

This lab connects `tf.einsum` directly to the formulas you studied in the prereq readings.
Every section is labeled with the reading it implements, so you can see exactly where the
math lives in the code.

| Section | Reading | Formula implemented |
|---------|---------|---------------------|
| 1 | **r2** | Transpose, sum, trace, diagonal |
| 2 | **r2** | Inner product, outer product, matrix–vector, matmul, Gram matrix |
| 3 | — | Batch operations |
| 4 | **r5** | Quadratic form $x^\top A x$, PD verification |
| 5 | **r5** | Spectral decomposition $A = \sum_k \lambda_k q_k q_k^\top$ |
| 6 | **r5** | Truncated SVD $A_r = \sum_{i=1}^r \sigma_i u_i v_i^\top$ (Eckart–Young) |
| 7 | **r5** | PCA: covariance → eigh → projection $Z = XQ_r$ |
| 8 | **r5+3DGS** | Batch covariance $\Sigma = R\,\mathrm{diag}(s^2)\,R^\top$ |
| 9 | — | EinsumDense Keras layer (linear map as einsum) |
| 10 | **r5** | GradientTape through $x^\top A x$ |
| 11 | — | Multi-head attention |
| 12 | — | `@tf.function` compilation benchmark |

The einsum **notation string** is identical across NumPy, PyTorch, TensorFlow, and JAX — learning it once gives you a portable vocabulary for all four.

In [ ]:
import tensorflow as tf
import numpy as np
import time

tf.random.set_seed(42)
np.random.seed(42)
print(f'TensorFlow {tf.__version__}')
print(f'GPU available: {len(tf.config.list_physical_devices("GPU")) > 0}')

## Notation Primer

| einsum string | Meaning | r2 / r5 reference |
|--------------|---------|-------------------|
| `'ij->ji'` | Transpose: $B_{ji} = A_{ij}$ | r2 §1 |
| `'ij->'` | Sum all elements | r2 §2 |
| `'ii->'` | Trace: $\sum_i A_{ii}$ | r2 §3 |
| `'i,i->'` | Dot product: $a^\top b = \sum_i a_i b_i$ | r2 §4 |
| `'i,j->ij'` | Outer product: $C_{ij} = a_i b_j$ | r2 §4 |
| `'ij,j->i'` | Matrix–vector: $(Ax)_i = \sum_j A_{ij} x_j$ | r2 §5 |
| `'ij,jk->ik'` | Matrix multiply: $C_{ik} = \sum_j A_{ij} B_{jk}$ | r2 §5 |
| `'ni,nj->ij'` | Gram matrix: $X^\top X$ (sample covariance kernel) | r2 §5, r5 §6 |
| `'i,ij,j->'` | Quadratic form: $x^\top A x$ | r5 §3 |
| `'k,ik,jk->ij'` | Spectral decomp: $A = \sum_k \lambda_k q_k q_k^\top$ | r5 §4 |
| `'r,mr,rn->mn'` | Truncated SVD: $A_r = \sum_i \sigma_i u_i v_i^\top$ | r5 §5 |
| `'nd,dr->nr'` | PCA projection: $Z = X Q_r$ | r5 §6 |
| `'nik,nk,njk->nij'` | Batch covariance: $\Sigma_n = R_n\,\mathrm{diag}(s_n^2)\,R_n^\top$ | r5 §4 + 3DGS |

---
## Section 1 — Unary Operations &nbsp; `📖 r2`

Reading r2 defines transpose ($B_{ji} = A_{ij}$), trace ($\text{tr}(A) = \sum_i A_{ii}$),
and the diagonal vector. Einsum expresses all of them with a single letter-rearrangement rule:
any index that appears on the **left only** is summed over; any index on the **right** is kept.

In [ ]:
A = tf.constant(np.arange(1, 17, dtype=np.float32).reshape(4, 4))

def chk(name, a, b, atol=1e-5):
    ok = np.allclose(a.numpy() if hasattr(a, 'numpy') else a,
                     b.numpy() if hasattr(b, 'numpy') else b, atol=atol)
    print(f'  {name:<40} match={ok}')

# r2 §1: Transpose  B_ji = A_ij
chk('Transpose      ij->ji', tf.einsum('ij->ji', A), tf.transpose(A))

# Sum all entries
chk('Sum all        ij->',   tf.einsum('ij->', A),   tf.reduce_sum(A))

# Row sums
chk('Row sums       ij->i',  tf.einsum('ij->i', A),  tf.reduce_sum(A, axis=1))

# r2 §3: Trace  tr(A) = sum_i A_ii
chk('Trace          ii->',   tf.einsum('ii->', A),   tf.linalg.trace(A))

# Diagonal vector
chk('Diagonal       ii->i',  tf.einsum('ii->i', A),  tf.linalg.diag_part(A))

print()
print(f'A =\n{A.numpy()}')
print(f'\ntr(A) = {tf.einsum("ii->", A).numpy():.0f}  (sum of diagonal = 1+6+11+16 = 34)')

---
## Section 2 — Binary Products &nbsp; `📖 r2`

Reading r2 §4–5 introduces inner product, outer product, and matrix multiplication.  
The **Gram matrix** $G = X^\top X$ (where $G_{jk} = \sum_n X_{nj} X_{nk}$) reappears in
Section 7 as the sample covariance.

In [ ]:
x = tf.constant([1., 2., 3.])
y = tf.constant([4., 5., 6.])
M = tf.constant(np.random.randn(3, 4).astype(np.float32))
N = tf.constant(np.random.randn(4, 5).astype(np.float32))
P = tf.constant(np.random.randn(3, 4).astype(np.float32))
X_gram = tf.constant(np.random.randn(10, 4).astype(np.float32))  # 10 samples, 4 features

# r2 §4: Inner product  a'b = sum_i a_i b_i
chk('Inner product  i,i->',    tf.einsum('i,i->', x, y),    tf.tensordot(x, y, axes=1))

# r2 §4: Outer product  C_ij = a_i b_j
outer_ref = tf.reshape(x, (-1, 1)) * tf.reshape(y, (1, -1))
chk('Outer product  i,j->ij',  tf.einsum('i,j->ij', x, y),  outer_ref)

# r2 §5: Matrix-vector  (Mv)_i = sum_j M_ij v_j
chk('Matrix–vector  ij,j->i',  tf.einsum('ij,j->i', M, x),  tf.linalg.matvec(M, x))

# r2 §5: Matrix multiply  C_ik = sum_j A_ij B_jk
chk('Matrix multiply ij,jk->ik', tf.einsum('ij,jk->ik', M, N), tf.matmul(M, N))

# Frobenius inner product  <A,B> = sum_{ij} A_ij B_ij
chk('Frobenius      ij,ij->',  tf.einsum('ij,ij->', M, P),  tf.reduce_sum(M * P))

# r2 §5 / r5 §6: Gram matrix  G_jk = sum_n X_nj X_nk  (= X'X)
G_ein = tf.einsum('ni,nj->ij', X_gram, X_gram)  # shape (4,4)
G_ref = tf.matmul(tf.transpose(X_gram), X_gram)
chk('Gram matrix    ni,nj->ij', G_ein, G_ref)
print(f'\nGram matrix shape: {G_ein.shape}  (features × features)')

---
## Section 3 — Batch Operations

Prefixing a shared batch index `b` to any binary contraction vectorises it across a batch
without a Python loop.  These patterns appear in deep learning wherever we process
multiple samples simultaneously.

In [ ]:
B, m, k, n = 8, 4, 6, 5
As = tf.constant(np.random.randn(B, m, k).astype(np.float32))
Bs = tf.constant(np.random.randn(B, k, n).astype(np.float32))
Sq = tf.constant(np.random.randn(B, 4, 4).astype(np.float32))
U  = tf.constant(np.random.randn(B, 3).astype(np.float32))
V  = tf.constant(np.random.randn(B, 3).astype(np.float32))

def bchk(name, a, b):
    ok = np.allclose(a.numpy(), b.numpy(), atol=1e-5)
    print(f'  {name:<40} shape={tuple(a.shape)}  match={ok}')

# Batch matmul
bchk('Batch matmul   bik,bkj->bij', tf.einsum('bik,bkj->bij', As, Bs), tf.matmul(As, Bs))

# Batch trace
bchk('Batch trace    bii->b',       tf.einsum('bii->b', Sq),            tf.linalg.trace(Sq))

# Batch outer product  C_bij = u_bi * v_bj
bop_ref = tf.matmul(tf.expand_dims(U, 2), tf.expand_dims(V, 1))
bchk('Batch outer    bi,bj->bij',   tf.einsum('bi,bj->bij', U, V),      bop_ref)

# Batch diagonal
bchk('Batch diagonal bii->bi',      tf.einsum('bii->bi', Sq),           tf.linalg.diag_part(Sq))

---
## Section 4 — Quadratic Forms &nbsp; `📖 r5`

Reading r5 §3 defines the quadratic form $f(x) = x^\top A x = \sum_{i,j} x_i A_{ij} x_j$.  
A matrix is **positive definite** (PD) iff $x^\top A x > 0$ for all $x \neq 0$, which r5
characterises as: all eigenvalues $\lambda_k > 0$.  We verify both conditions below.

r5 also gives two spectral identities:
$$\text{tr}(A) = \sum_k \lambda_k \qquad \det(A) = \prod_k \lambda_k$$

In [ ]:
rng = np.random.default_rng(0)
d = 5

# Build a PD matrix: A = F'F + dI
F = rng.standard_normal((d + 3, d)).astype(np.float32)
A_pd = tf.constant(F.T @ F + d * np.eye(d, dtype=np.float32))  # guaranteed PD

# Build a symmetric but indefinite matrix
F2 = rng.standard_normal((d, d)).astype(np.float32)
A_indef = tf.constant((F2 + F2.T).astype(np.float32))  # symmetric, may have negative eigs

x_test = tf.constant(rng.standard_normal(d).astype(np.float32))

# Quadratic form: x'Ax = einsum('i,ij,j->', x, A, x)
qf_pd    = tf.einsum('i,ij,j->', x_test, A_pd,    x_test)
qf_indef = tf.einsum('i,ij,j->', x_test, A_indef, x_test)

# Eigenvalues (r5 §4: eigh returns sorted ascending for symmetric/Hermitian)
lam_pd    = tf.linalg.eigvalsh(A_pd)     # all positive → PD
lam_indef = tf.linalg.eigvalsh(A_indef)  # may include negatives → indefinite

print('Quadratic form & positive-definiteness')
print(f'  A_pd:    x\'Ax = {qf_pd.numpy():.4f}  (positive ✓)')
print(f'  A_indef: x\'Ax = {qf_indef.numpy():.4f}')
print()
print(f'  A_pd eigenvalues    : {np.round(lam_pd.numpy(), 3)}  (all > 0 → PD)')
print(f'  A_indef eigenvalues : {np.round(lam_indef.numpy(), 3)}')
print()

# r5: tr(A) = sum of eigenvalues,  det(A) = product of eigenvalues
tr_ein  = tf.einsum('ii->', A_pd).numpy()
tr_eig  = tf.reduce_sum(lam_pd).numpy()
det_np  = np.linalg.det(A_pd.numpy())
det_eig = tf.reduce_prod(lam_pd).numpy()
print('Spectral identities (r5)')
print(f'  tr(A) via einsum       = {tr_ein:.4f}')
print(f'  tr(A) via sum(eigs)    = {tr_eig:.4f}  match={np.isclose(tr_ein, tr_eig, atol=1e-4)}')
print(f'  det(A) via np.linalg   = {det_np:.4f}')
print(f'  det(A) via prod(eigs)  = {det_eig:.4f}  match={np.isclose(det_np, det_eig, rtol=1e-4)}')

---
## Section 5 — Spectral Decomposition &nbsp; `📖 r5`

Reading r5 §4 (Spectral Theorem) states that any real symmetric matrix has the decomposition
$$A = Q \Lambda Q^\top = \sum_{k=1}^n \lambda_k q_k q_k^\top$$
where $Q$ is orthonormal ($Q^\top Q = I$) and $\Lambda = \text{diag}(\lambda_1, \ldots, \lambda_n)$.

The **einsum string** `'k,ik,jk->ij'` computes exactly this sum of rank-1 terms:
$$A_{ij} = \sum_k \lambda_k \, (q_k)_i \, (q_k)_j$$

Truncating to $r < n$ terms gives the best rank-$r$ approximation (Eckart–Young, shown in Section 6).

In [ ]:
rng = np.random.default_rng(1)
n = 6
F = rng.standard_normal((n + 2, n)).astype(np.float32)
A_sym = tf.constant((F.T @ F).astype(np.float32))  # symmetric PSD, rank n

# r5: eigendecomposition of symmetric matrix (eigh guarantees real, sorted ascending)
lam, Q = tf.linalg.eigh(A_sym)   # lam: (n,), Q: (n,n) — columns are eigenvectors

# Reconstruct A from spectral decomp: A = Q diag(lam) Q' = einsum('k,ik,jk->ij', lam, Q, Q)
A_recon = tf.einsum('k,ik,jk->ij', lam, Q, Q)
recon_err = tf.reduce_max(tf.abs(A_sym - A_recon)).numpy()

print('Spectral decomposition: A = Σ λ_k q_k q_k\'')
print(f'  Eigenvalues : {np.round(lam.numpy(), 4)}')
print(f'  Max reconstruction error: {recon_err:.2e}')
print(f'  Q orthonormal: {np.allclose((Q.numpy().T @ Q.numpy()), np.eye(n), atol=1e-5)}')
print()

# Individual rank-1 contributions and cumulative reconstruction error
# Sort eigenvalues descending for truncation experiment
order = tf.argsort(lam, direction='DESCENDING')
lam_d = tf.gather(lam, order)
Q_d   = tf.gather(Q, order, axis=1)

total_var = tf.reduce_sum(lam_d).numpy()
print(f'  {"Rank r":<8} {"Cumul var (%)":<18} {"||A - A_r||_F"}')
print('  ' + '-' * 44)
for r in range(1, n + 1):
    A_r = tf.einsum('k,ik,jk->ij', lam_d[:r], Q_d[:, :r], Q_d[:, :r])
    err = tf.norm(A_sym - A_r, ord='fro').numpy()
    cum_var = 100 * tf.reduce_sum(lam_d[:r]).numpy() / total_var
    print(f'  {r:<8} {cum_var:<18.2f} {err:.4f}')

---
## Section 6 — Truncated SVD & Eckart–Young &nbsp; `📖 r5`

Reading r5 §5 introduces the **Singular Value Decomposition** $A = U \Sigma V^\top$ and the
Eckart–Young theorem: the best rank-$r$ approximation in Frobenius norm is
$$A_r = \sum_{i=1}^r \sigma_i \, u_i \, v_i^\top$$
The einsum string `'r,mr,rn->mn'` builds this sum directly.

r5 also states the connection $\sigma_i = \sqrt{\lambda_i(A^\top A)}$, which we verify below.

In [ ]:
rng = np.random.default_rng(2)
m_svd, n_svd = 8, 5
A_rect = tf.constant(rng.standard_normal((m_svd, n_svd)).astype(np.float32))

# tf.linalg.svd returns (S, U, V) where A = U diag(S) V'
# Note: TF returns V (not V'), so the outer product is u_i @ v_i'
S_tf, U_tf, V_tf = tf.linalg.svd(A_rect)   # S: (k,), U: (m,k), V: (n,k)

print('SVD shape check')
print(f'  A: {A_rect.shape}, U: {U_tf.shape}, S: {S_tf.shape}, V: {V_tf.shape}')
print(f'  Singular values: {np.round(S_tf.numpy(), 4)}')

# Full reconstruction using einsum: A = einsum('r,mr,nr->mn', S, U, V)
# V_tf has shape (n,k) so we use 'nr' (not 'rn')
A_full = tf.einsum('r,mr,nr->mn', S_tf, U_tf, V_tf)
print(f'\n  Full reconstruction error: {tf.norm(A_rect - A_full, ord="fro").numpy():.2e}')

# r5: sigma_i = sqrt(lambda_i(A'A)) — verify
AtA = tf.matmul(tf.transpose(A_rect), A_rect)
eigs_AtA = tf.linalg.eigvalsh(AtA)                # sorted ascending
sigma_from_eig = tf.sqrt(tf.maximum(eigs_AtA, 0)) # sqrt(eigenvalues), clamp negatives from numerics
sigma_from_eig = tf.reverse(sigma_from_eig, axis=[0])  # descending to match SVD order
print(f'\n  sigma via SVD      : {np.round(S_tf.numpy(), 4)}')
print(f'  sigma via sqrt(eig): {np.round(sigma_from_eig.numpy(), 4)}')
print(f'  Match: {np.allclose(S_tf.numpy(), sigma_from_eig.numpy(), atol=1e-4)}')

# Eckart-Young: truncated rank-r approximation
r_max = min(m_svd, n_svd)
print(f'\n  {"Rank r":<8} {"||A - A_r||_F":<18} {"Predicted (σ_{r+1}..σ_k)"}')
print('  ' + '-' * 52)
sigma_sq = S_tf.numpy() ** 2
for r in range(1, r_max + 1):
    A_r = tf.einsum('r,mr,nr->mn', S_tf[:r], U_tf[:, :r], V_tf[:, :r])
    err = tf.norm(A_rect - A_r, ord='fro').numpy()
    predicted = np.sqrt(sigma_sq[r:].sum()) if r < r_max else 0.0
    print(f'  {r:<8} {err:<18.4f} {predicted:.4f}')

---
## Section 7 — PCA Pipeline &nbsp; `📖 r5`

Reading r5 §6 derives PCA from the spectral decomposition of the sample covariance matrix.
The four steps, each an einsum, are:

1. **Centre** the data: $\tilde{X} = X - \bar{x}^\top$
2. **Covariance**: $C = \tfrac{1}{N-1} \tilde{X}^\top \tilde{X}$ — einsum `'ni,nj->ij'`
3. **Eigendecompose**: $C = Q \Lambda Q^\top$ via `tf.linalg.eigh`
4. **Project**: $Z = \tilde{X} Q_r$ — einsum `'nd,dr->nr'`

The principal components in $Z$ are guaranteed to be **uncorrelated** (covariance matrix is diagonal).

In [ ]:
rng = np.random.default_rng(3)
N_pca, d_pca, r_pca = 200, 8, 3

# Simulate correlated data
true_cov = rng.standard_normal((d_pca, d_pca)).astype(np.float32)
true_cov = true_cov.T @ true_cov + np.eye(d_pca, dtype=np.float32)
L = np.linalg.cholesky(true_cov)
X_raw = (rng.standard_normal((N_pca, d_pca)) @ L.T).astype(np.float32)
X_raw = tf.constant(X_raw)

# Step 1: Centre
X_c = X_raw - tf.reduce_mean(X_raw, axis=0, keepdims=True)

# Step 2: Sample covariance using einsum (r5: C = (1/(N-1)) X'X)
C = tf.einsum('ni,nj->ij', X_c, X_c) / (N_pca - 1)   # (d,d)
print(f'Covariance shape: {C.shape}')

# Step 3: Eigendecompose (eigh: ascending order)
lam_pca, Q_pca = tf.linalg.eigh(C)

# Reorder descending (largest variance first)
order = tf.argsort(lam_pca, direction='DESCENDING')
lam_pca = tf.gather(lam_pca, order)
Q_pca   = tf.gather(Q_pca, order, axis=1)

# Step 4: Project to r principal components (r5: Z = X Q_r)
Q_r = Q_pca[:, :r_pca]                                 # (d, r)
Z   = tf.einsum('nd,dr->nr', X_c, Q_r)                 # (N, r)
print(f'Projection shape: {Z.shape}  ({N_pca} samples, {r_pca} PCs)')

# Verify: principal components are uncorrelated (off-diagonal covariance ≈ 0)
C_Z = tf.einsum('ni,nj->ij', Z, Z) / (N_pca - 1)       # (r, r)
off_diag = tf.abs(C_Z - tf.linalg.diag(tf.linalg.diag_part(C_Z)))
print(f'Max off-diagonal in PC covariance: {tf.reduce_max(off_diag).numpy():.2e}  (≈0 → uncorrelated ✓)')

# Variance explained
total_var = tf.reduce_sum(lam_pca).numpy()
print(f'\n  {"PC":<6} {"Eigenvalue":<14} {"Var explained %"}')
print('  ' + '-' * 36)
cumul = 0
for i in range(r_pca):
    pct = 100 * lam_pca[i].numpy() / total_var
    cumul += pct
    print(f'  {i+1:<6} {lam_pca[i].numpy():<14.4f} {pct:.2f}%  (cumul {cumul:.2f}%)')

print()
print('3DGS link: the eigenvectors Q define the rotation R of each Gaussian;')
print('the eigenvalues λ_k correspond to the squared scales s_k² — see Section 8.')

---
## Section 8 — Batch Covariance & 3DGS Connection &nbsp; `📖 r5 + 3DGS`

Reading r5 §4 shows that any PSD matrix can be written $\Sigma = Q \Lambda Q^\top$.
In 3D Gaussian Splatting each Gaussian has a covariance matrix parameterised as
$$\Sigma_n = R_n \, \text{diag}(s_n^2) \, R_n^\top$$
where $R_n \in SO(3)$ is a rotation matrix and $s_n \in \mathbb{R}^3_{>0}$ is a scale vector.
This is **exactly** the spectral decomposition from r5 with $Q = R_n$ and $\Lambda = \text{diag}(s_n^2)$.

The einsum string `'nik,nk,njk->nij'` computes this for all $N$ Gaussians simultaneously:
$$\Sigma_{n,ij} = \sum_k R_{n,ik} \, s_{n,k}^2 \, R_{n,jk}$$

In [ ]:
from scipy.spatial.transform import Rotation   # for generating valid SO(3) rotations

rng = np.random.default_rng(4)
N_gauss = 500   # number of 3DGS Gaussians
dim     = 3

# Random rotation matrices in SO(3) for each Gaussian
R_np = Rotation.random(N_gauss, random_state=42).as_matrix().astype(np.float32)  # (N,3,3)
R    = tf.constant(R_np)

# Random positive scales (log-normally distributed, as in 3DGS)
s   = tf.constant(rng.lognormal(0, 0.5, (N_gauss, dim)).astype(np.float32))
s2  = s ** 2   # squared scales = eigenvalues

# Batch covariance via einsum: Sigma_n = R_n diag(s_n^2) R_n'
# 'nik,nk,njk->nij':
#   n = Gaussian index, i/j = spatial dims (output), k = eigenvector index (summed)
Sigma = tf.einsum('nik,nk,njk->nij', R, s2, R)   # (N,3,3)

print(f'Batch covariance shape: {Sigma.shape}')

# Verify symmetry: Sigma_n = Sigma_n'
sym_err = tf.reduce_max(tf.abs(Sigma - tf.transpose(Sigma, perm=[0,2,1])))
print(f'Max symmetry error: {sym_err.numpy():.2e}  (Σ = Σ\' ✓)')

# Verify PSD: all eigenvalues >= 0
eigs_Sigma = tf.linalg.eigvalsh(Sigma)   # (N,3), ascending
min_eig    = tf.reduce_min(eigs_Sigma).numpy()
print(f'Min eigenvalue across all Gaussians: {min_eig:.6f}  (>=0 → PSD ✓)')

# Verify eigenvalues recover s^2 (up to sorting)
# eigh returns ascending; sort s2 ascending for comparison
s2_sorted   = tf.sort(s2, axis=1)          # (N,3) ascending
eig_match   = np.allclose(eigs_Sigma.numpy(), s2_sorted.numpy(), atol=1e-4)
print(f'Eigenvalues of Σ_n == s_n² (sorted): {eig_match}')

print()
print('Structure comparison:')
print('  r5 spectral:  A    = Q   diag(λ)  Q\'')
print('  3DGS covar:   Σ_n  = R_n diag(s²) R_n\'')
print('  einsum:       nik,nk,njk->nij  (batch version of k,ik,jk->ij)')

---
## Section 9 — EinsumDense Keras Layer

A fully-connected layer computes a **linear map** $y = Wx + b$ — which is the matrix–vector product
from r2 §5.  Implemented with `tf.einsum('bd,dk->bk', X, W)`, weights declared with `add_weight`
are tracked automatically by Keras; gradients flow through the einsum call without any special
handling.

In [ ]:
class EinsumDense(tf.keras.layers.Layer):
    """Dense layer: y = einsum('bd,dk->bk', x, W) + b.

    Equivalent to tf.keras.layers.Dense but exposes the einsum string explicitly.
    This is the r2 §5 matrix-vector product: (Wx)_k = sum_d W_dk x_d,
    batched over a mini-batch dimension b.
    """

    def __init__(self, units, **kwargs):
        super().__init__(**kwargs)
        self.units = units

    def build(self, input_shape):
        d_in = input_shape[-1]
        self.W = self.add_weight(
            name='kernel', shape=(d_in, self.units),
            initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight(
            name='bias', shape=(self.units,),
            initializer='zeros', trainable=True)

    def call(self, x):
        # r2 §5: matrix-vector product, batched
        return tf.einsum('bd,dk->bk', x, self.W) + self.b


rng_np = np.random.default_rng(5)
batch, d_in2, d_h, d_out = 32, 16, 64, 10
X_test = rng_np.standard_normal((batch, d_in2)).astype(np.float32)

model_ein = tf.keras.Sequential([
    EinsumDense(d_h, name='hidden'),
    tf.keras.layers.ReLU(),
    EinsumDense(d_out, name='output'),
])
model_dense = tf.keras.Sequential([
    tf.keras.layers.Dense(d_h, activation='relu'),
    tf.keras.layers.Dense(d_out),
])

out_ein   = model_ein(X_test)
out_dense = model_dense(X_test)
print(f'EinsumDense output shape : {out_ein.shape}')
print(f'Dense output shape       : {out_dense.shape}')
assert model_ein.count_params() == model_dense.count_params()
print(f'Parameter count matches  : {model_ein.count_params()} ✓')

# Quick training verification
model_ein.compile(optimizer='adam', loss='mse')
y_fake = rng_np.standard_normal((batch, d_out)).astype(np.float32)
history = model_ein.fit(X_test, y_fake, epochs=5, verbose=0)
print(f'Training losses (5 epochs): {[f"{l:.4f}" for l in history.history["loss"]]}')
print('Gradients flow through EinsumDense ✓')

---
## Section 10 — GradientTape Through Einsum &nbsp; `📖 r5`

Reading r5 §3 states that for a symmetric matrix $A$:
$$\nabla_x \, x^\top A x = 2 A x$$

`tf.GradientTape` records the forward pass through `tf.einsum('i,ij,j->', x, A, x)` and
computes the exact analytical gradient — matching the formula from r5.

In [ ]:
rng_np = np.random.default_rng(6)
d_grad = 5

# Symmetric PD matrix
F_grad = rng_np.standard_normal((d_grad, d_grad)).astype(np.float32)
A_grad = tf.constant((F_grad.T @ F_grad + d_grad * np.eye(d_grad)).astype(np.float32))
x_var  = tf.Variable(tf.constant(rng_np.standard_normal(d_grad).astype(np.float32)))

# Gradient of quadratic form f(x) = x'Ax
with tf.GradientTape() as tape:
    qf = tf.einsum('i,ij,j->', x_var, A_grad, x_var)

grad_tape     = tape.gradient(qf, x_var)                    # automatic
grad_analytic = 2.0 * tf.linalg.matvec(A_grad, x_var)      # r5: ∇(x'Ax) = 2Ax for symmetric A

print('Gradient of quadratic form x\'Ax')
print(f'  f(x) = x\'Ax              = {qf.numpy():.6f}')
print(f'  GradientTape gradient    = {np.round(grad_tape.numpy(), 4)}')
print(f'  Analytic gradient 2Ax   = {np.round(grad_analytic.numpy(), 4)}')
print(f'  Match                    = {np.allclose(grad_tape.numpy(), grad_analytic.numpy(), atol=1e-5)}')

print()

# Second-order: Hessian of x'Ax should equal 2A
with tf.GradientTape() as outer:
    with tf.GradientTape() as inner:
        qf2 = tf.einsum('i,ij,j->', x_var, A_grad, x_var)
    g = inner.gradient(qf2, x_var)
H_tape = outer.jacobian(g, x_var)    # Hessian via nested tape
H_analytic = 2.0 * A_grad
print(f'  Hessian matches 2A = {np.allclose(H_tape.numpy(), H_analytic.numpy(), atol=1e-4)}')

---
## Section 11 — Multi-Head Attention via Einsum

The two core einsum patterns in scaled dot-product attention, followed by an output projection.

In [ ]:
def einsum_mha(Q_in, K_in, V_in, W_o):
    """
    Multi-head attention via einsum.
    Q_in, K_in, V_in : (batch, heads, seq, d_head)
    W_o              : (heads, d_head, d_model)
    Returns          : (batch, seq, d_model)
    """
    d_head = tf.cast(tf.shape(Q_in)[-1], tf.float32)

    # QK scores: S_{b,h,i,j} = Σ_d Q_{b,h,i,d} K_{b,h,j,d}
    scores   = tf.einsum('bhid,bhjd->bhij', Q_in, K_in) * (d_head ** -0.5)
    weights  = tf.nn.softmax(scores, axis=-1)

    # Weighted values: O_{b,h,i,d} = Σ_j w_{b,h,i,j} V_{b,h,j,d}
    attended = tf.einsum('bhij,bhjd->bhid', weights, V_in)

    # Output projection: out_{b,i,m} = Σ_{h,d} O_{b,h,i,d} W_{h,d,m}
    out = tf.einsum('bhid,hdm->bim', attended, W_o)
    return out, weights


Bsz, H, S, Dh, Dm = 2, 4, 8, 16, 64
Q_mha = tf.random.normal((Bsz, H, S, Dh))
K_mha = tf.random.normal((Bsz, H, S, Dh))
V_mha = tf.random.normal((Bsz, H, S, Dh))
W_o   = tf.random.normal((H, Dh, Dm))

out_mha, attn_w = einsum_mha(Q_mha, K_mha, V_mha, W_o)
print(f'Output shape     : {out_mha.shape}  (batch, seq, d_model)')
print(f'Attention weights: {attn_w.shape}')
print(f'Weights sum to 1 : {np.allclose(tf.reduce_sum(attn_w, axis=-1).numpy(), 1.0, atol=1e-6)}')

# Compare QK step against tf.matmul
scores_ref = tf.matmul(Q_mha, K_mha, transpose_b=True) * (Dh ** -0.5)
scores_ein = tf.einsum('bhid,bhjd->bhij', Q_mha, K_mha) * (Dh ** -0.5)
print(f'QK scores match tf.matmul: {np.allclose(scores_ein.numpy(), scores_ref.numpy(), atol=1e-5)}')

---
## Section 12 — `@tf.function` Compilation Benchmark

Wrapping einsum operations in `@tf.function` traces them to XLA/TF graph IR, enabling
operation fusion and layout optimisation. Speedup is typically 2–8× for moderate tensor sizes.

In [ ]:
def bench_tf(fn, *args, warmup=5, iters=50):
    for _ in range(warmup):
        fn(*args)
    t0 = time.perf_counter()
    for _ in range(iters):
        _ = fn(*args)
    return (time.perf_counter() - t0) / iters * 1e3  # ms per call


def attn_eager(Q, K, V):
    d = tf.cast(tf.shape(Q)[-1], tf.float32)
    s = tf.einsum('bhid,bhjd->bhij', Q, K) * (d ** -0.5)
    w = tf.nn.softmax(s, axis=-1)
    return tf.einsum('bhij,bhjd->bhid', w, V)


@tf.function
def attn_compiled(Q, K, V):
    d = tf.cast(tf.shape(Q)[-1], tf.float32)
    s = tf.einsum('bhid,bhjd->bhij', Q, K) * (d ** -0.5)
    w = tf.nn.softmax(s, axis=-1)
    return tf.einsum('bhij,bhjd->bhid', w, V)


Bsz2, H2, S2, Dh2 = 16, 8, 64, 32
Q2 = tf.random.normal((Bsz2, H2, S2, Dh2))
K2 = tf.random.normal((Bsz2, H2, S2, Dh2))
V2 = tf.random.normal((Bsz2, H2, S2, Dh2))

t_eager    = bench_tf(attn_eager,    Q2, K2, V2)
t_compiled = bench_tf(attn_compiled, Q2, K2, V2)

print(f'Attention ({Bsz2}×{H2} heads×{S2} seq×{Dh2} d_head)')
print(f'  Eager    : {t_eager:.3f} ms')
print(f'  Compiled : {t_compiled:.3f} ms')
print(f'  Speedup  : {t_eager / t_compiled:.2f}x')

# Verify numerical equivalence
out_e = attn_eager(Q2, K2, V2)
out_c = attn_compiled(Q2, K2, V2)
print(f'\nNumerical match (eager vs compiled): {np.allclose(out_e.numpy(), out_c.numpy(), atol=1e-5)}')

---
## Summary — Reading Formulas → Einsum Strings

| Reading formula | Einsum string | Section |
|----------------|--------------|--------|
| $B_{ji} = A_{ij}$ (r2 transpose) | `'ij->ji'` | 1 |
| $\text{tr}(A) = \sum_i A_{ii}$ (r2 trace) | `'ii->'` | 1 |
| $a^\top b = \sum_i a_i b_i$ (r2 inner product) | `'i,i->'` | 2 |
| $C_{ij} = a_i b_j$ (r2 outer product) | `'i,j->ij'` | 2 |
| $C_{ik} = \sum_j A_{ij} B_{jk}$ (r2 matmul) | `'ij,jk->ik'` | 2 |
| $G_{jk} = \sum_n X_{nj} X_{nk} = [X^\top X]_{jk}$ (r2 Gram) | `'ni,nj->ij'` | 2, 7 |
| $x^\top A x = \sum_{i,j} x_i A_{ij} x_j$ (r5 quadratic form) | `'i,ij,j->'` | 4, 10 |
| $A = \sum_k \lambda_k q_k q_k^\top$ (r5 spectral decomp) | `'k,ik,jk->ij'` | 5 |
| $A_r = \sum_{i=1}^r \sigma_i u_i v_i^\top$ (r5 Eckart–Young) | `'r,mr,nr->mn'` | 6 |
| $Z = X Q_r$ (r5 PCA projection) | `'nd,dr->nr'` | 7 |
| $\Sigma_n = R_n \, \text{diag}(s_n^2) \, R_n^\top$ (r5 + 3DGS) | `'nik,nk,njk->nij'` | 8 |

**TensorFlow-specific takeaways:**
- `tf.einsum` is fully differentiable — `GradientTape` computes exact gradients through any contraction.
- `@tf.function` fuses einsum ops into an XLA graph, typically 2–4× faster on GPU.
- `tf.linalg.eigh` / `tf.linalg.svd` decompose matrices; their outputs feed directly into einsum strings that reconstruct or project.
- The einsum notation string is **identical** across NumPy, PyTorch, TensorFlow, and JAX.